## Install Dependencies

In [1]:
!pip install -qU peft trl transformers datasets accelerate bitsandbytes codecarbon psutil gputil

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.5/380.5 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.1 MB/s eta 0:00:00


## Imports & Setup (+ Huggingface Token Setup)

In [2]:
import json, time, os, gc, threading
import torch
import psutil
import numpy as np
from datetime import datetime
from datasets import load_dataset
from transformers import AutoTokenizer
from codecarbon import EmissionsTracker

In [4]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [5]:
import warnings
warnings.filterwarnings("ignore")

## GPU (cuda) Info Preview + GPU Memory Utils

In [6]:
import torch, transformers

print("torch:", torch.__version__)
print("torch.cuda:", torch.version.cuda)
print("transformers:", transformers.__version__)
print("." * 50)

if torch.cuda.is_available():
    print("CUDA is available ✅")
    print("Device count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"\n--- GPU {i} ---")
        print("Name:", torch.cuda.get_device_name(i))
        print("Capability:", torch.cuda.get_device_capability(i))
        print("Total memory (GB):", torch.cuda.get_device_properties(i).total_memory / 1e9)
else:
    print("CUDA is NOT available ❌")

# GPU memory utils
def get_gpu_memory_mb(device=0):
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated(device) / 1024**2
    return 0.0

def get_vram_reserved_mb(device=0):
    if torch.cuda.is_available():
        return torch.cuda.memory_reserved(device) / 1024**2
    return 0.0

torch: 2.10.0+cu128
torch.cuda: 12.8
transformers: 5.8.1
..................................................
CUDA is available ✅
Device count: 1

--- GPU 0 ---
Name: Tesla T4
Capability: (7, 5)
Total memory (GB): 15.637086208


## 1. Load data & tokenizer

In [37]:
from datasets import load_dataset

RANDOM_STATE = 67 # for reproducibility

hf_dataset = load_dataset(
    "Salesforce/xlam-function-calling-60k",
    split="train",
    token=HF_TOKEN
)

def add_total_input_length(example):
    query_len = len(str(example["query"]))
    tools_len = len(str(example["tools"]))
    example["total_input_length"] = query_len + tools_len
    return example

hf_dataset = hf_dataset.map(add_total_input_length)

# keep only rows with total_input_length < 4096
filtered_dataset = hf_dataset.filter(
    lambda x: x["total_input_length"] < 4096
)

# Dataset ~ get: 10,000 samples
subset = filtered_dataset.shuffle(seed=RANDOM_STATE).select(range(10000))

# TEST SET ~ 10%
split_set = subset.train_test_split(test_size=1000, seed=RANDOM_STATE)
test_dataset = split_set['test']
train_dataset = split_set['train'] # TRAIN SET (SFT) ~ 90%

print(f"Test Dataset: {len(test_dataset)} examples")
print(f"SFT Dataset: {len(train_dataset)} examples")

Test Dataset: 1000 examples
SFT Dataset: 9000 examples


In [51]:
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

## 2. Dataset formatting

In [39]:
def format_tool_calling(row):
    system_prompt = (
        "You are a helpful assistant with access to the following functions. "
        f"Use them if required:\n{row['tools']}\n\n"
        "Respond strictly with a JSON array of function calls."
    )
    messages = [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": row["query"]},
        {"role": "assistant", "content": row["answers"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False)}

train_dataset = train_dataset.map(
    format_tool_calling, remove_columns=train_dataset.column_names)
print(f"Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Train size: 9000 | Test size: 1000


## 3. Instrumented inference + benchmark

In [56]:
from tqdm import tqdm
import re

##### HELPERS #####
_THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)

def _strip_thinking_block(text: str) -> str:
    """Remove <think>…</think> emitted by Qwen3 in thinking mode (or residual tags)."""
    return _THINK_RE.sub("", text).strip()

def safe_parse_json(value, fallback=None):
    if value is None: return fallback
    if isinstance(value, (dict, list)): return value
    if isinstance(value, str):
        try: return json.loads(value)
        except (json.JSONDecodeError, ValueError): return fallback
    return fallback

def safe_get_args(call: dict) -> dict:
    for key in ("arguments", "parameters", "args", "input"):
        if key in call:
            result = safe_parse_json(call[key], fallback={})
            if isinstance(result, dict): return result
    return {}

def build_prompt(row: dict) -> str:
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant with access to the following functions. "
            f"Use them if required:\n{row['tools']}\n\n"
            "Respond strictly with a JSON array of function calls."
        )},
        {"role": "user", "content": row["query"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,   # disables <think>…</think> block for fast tool-call inference
    )


##### instrumented batched inference  #####
def run_batch_inference(model_, rows: list[dict], batch_size: int = 8) -> list[dict]:
    """
    Runs inference over a list of {query, tools} dicts in batches.
    Returns a list of per-sample result dicts with response + perf stats.
    """
    # tokenizer must pad to the left for decoder-only batch generation
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    all_results = []
    proc = psutil.Process(os.getpid())

    for batch_start in tqdm(range(0, len(rows), batch_size)):
        batch = rows[batch_start : batch_start + batch_size]
        prompts = [build_prompt(r) for r in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        ).to(model_.device)

        input_len = inputs["input_ids"].shape[1]

        ram_before  = proc.memory_info().rss / 1024**2
        vram_before = get_gpu_memory_mb()
        cpu_pct     = psutil.cpu_percent(interval=None)

        # TTFT: generate 1 token for the whole batch
        ttft_start = time.perf_counter()
        with torch.no_grad():
            model_.generate(**inputs, max_new_tokens=1, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        ttft_s = time.perf_counter() - ttft_start

        # Full generation
        gen_start = time.perf_counter()
        with torch.no_grad():
            outputs = model_.generate(
                **inputs,
                max_new_tokens=1024,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        batch_time_s = time.perf_counter() - gen_start

        ram_after  = proc.memory_info().rss / 1024**2
        vram_after = get_gpu_memory_mb()

        # Decode each sample individually (strip the input prefix)
        for i, output_ids in enumerate(outputs):
            new_ids    = output_ids[input_len:]
            tokens_gen = int((new_ids != tokenizer.pad_token_id).sum())
            response_text = tokenizer.decode(new_ids, skip_special_tokens=True)
            response_text = _strip_thinking_block(response_text)

            all_results.append({
                "response_text":    response_text,
                "total_time_s":     round(batch_time_s / len(batch), 4),  # per-sample share
                "ttft_s":           round(ttft_s / len(batch), 4),
                "tokens_generated": tokens_gen,
                "tokens_per_sec":   round(tokens_gen / (batch_time_s / len(batch)), 2)
                                    if batch_time_s > 0 else 0.0,
                "vram_used_mb":     round((vram_after - vram_before) / len(batch), 2),
                "vram_peak_mb":     round(get_vram_reserved_mb(), 2),
                "ram_delta_mb":     round((ram_after - ram_before) / len(batch), 2),
                "cpu_percent":      round(cpu_pct, 1),
            })

    return all_results

In [57]:
##### BENCHMARK (BATCHED) #####

def benchmark(model_, test_ds, label="Model", batch_size=8,
              save_json=True, json_path="benchmark_results.json") -> dict:

    # Filter valid rows up front
    valid_rows = [
        r for r in test_ds
        if isinstance(r.get("query"), str) and isinstance(r.get("tools"), str)
    ]
    n_total = len(valid_rows)
    print(f"[{label}] Running batched inference — "
          f"{n_total} samples, batch_size={batch_size}")

    co2_kg, tracker = None, None
    tracker = EmissionsTracker(
        project_name=f"benchmark_{label.replace(' ', '_')}",
        log_level="error", save_to_file=False)
    tracker.start()

    wall_start = time.time()

    all_results = run_batch_inference(model_, valid_rows, batch_size=batch_size)

    wall_total = time.time() - wall_start
    if tracker: co2_kg = tracker.stop()

    json_valid = name_match = args_exact = args_keys_match = 0
    n = 0  # rows that have scorable ground truth

    perf_keys = ["total_time_s", "ttft_s", "tokens_per_sec",
                 "vram_used_mb", "vram_peak_mb", "ram_delta_mb",
                 "cpu_percent", "tokens_generated"]
    perf_accum = {k: [] for k in perf_keys}

    for row, res in zip(valid_rows, all_results):
        for k in perf_keys:
            perf_accum[k].append(res[k])

        pred = safe_parse_json(res["response_text"], fallback=None)
        if not isinstance(pred, list): continue
        json_valid += 1

        gold = safe_parse_json(row.get("answers"), fallback=None)
        if not isinstance(gold, list): continue

        gold_calls = [c for c in gold if isinstance(c, dict)]
        pred_calls = [c for c in pred if isinstance(c, dict)]
        if not gold_calls: continue
        n += 1

        gold_names = {c.get("name") for c in gold_calls if c.get("name")}
        pred_names = {c.get("name") for c in pred_calls if c.get("name")}
        if gold_names and gold_names == pred_names: name_match += 1

        g_args = safe_get_args(gold_calls[0])
        p_args = safe_get_args(pred_calls[0]) if pred_calls else {}
        if g_args.keys() == p_args.keys(): args_keys_match += 1
        if g_args == p_args:               args_exact += 1

    if n == 0:
        print(f"[{label}] No scorable rows found.")
        return {}

    def avg(lst): return round(float(np.mean(lst)), 4) if lst else 0.0
    def p95(lst): return round(float(np.percentile(lst, 95)), 4) if lst else 0.0

    results = {
        "label":         label,
        "timestamp":     datetime.utcnow().isoformat() + "Z",
        "total_samples": n,
        "batch_size":    batch_size,
        "accuracy": {
            "json_valid_pct":      round(100 * json_valid      / n, 1),
            "name_match_pct":      round(100 * name_match      / n, 1),
            "args_keys_match_pct": round(100 * args_keys_match / n, 1),
            "args_exact_pct":      round(100 * args_exact      / n, 1),
        },
        "performance": {
            "wall_total_s":          round(wall_total, 2),
            "avg_latency_s":         avg(perf_accum["total_time_s"]),
            "p95_latency_s":         p95(perf_accum["total_time_s"]),
            "avg_ttft_s":            avg(perf_accum["ttft_s"]),
            "avg_tokens_per_sec":    avg(perf_accum["tokens_per_sec"]),
            "avg_tokens_generated":  avg(perf_accum["tokens_generated"]),
            "avg_vram_delta_mb":     avg(perf_accum["vram_used_mb"]),
            "peak_vram_reserved_mb": avg(perf_accum["vram_peak_mb"]),
            "avg_ram_delta_mb":      avg(perf_accum["ram_delta_mb"]),
            "avg_cpu_percent":       avg(perf_accum["cpu_percent"]),
            "throughput_samples_per_sec": round(n / wall_total, 2),
        },
        "co2": {
            "emissions_kg":  round(co2_kg, 6) if co2_kg else None,
            "emissions_g":   round(co2_kg * 1000, 4) if co2_kg else None
        },
    }

    # pretty print
    print(f"\n{'='*52}")
    print(f"  {label}")
    print(f"{'='*52}")
    print("  ACCURACY")
    for k, v in results["accuracy"].items():
        print(f"    {k:<28}: {v} %")
    print("  PERFORMANCE")
    for k, v in results["performance"].items():
        print(f"    {k:<28}: {v}")
    if co2_kg:
        print(f"  CO₂  {results['co2']['emissions_g']} g CO₂eq")
    print(f"{'='*52}\n")

    if save_json:
        existing = []
        if os.path.exists(json_path):
            with open(json_path) as f:
                try: existing = json.load(f)
                except: existing = []
        existing.append(results)
        with open(json_path, "w") as f:
            json.dump(existing, f, indent=2)
        print(f"  ✅ Results appended → {json_path}")

    return results

In [58]:
##### single-call wrapper kept for ask_function() #####

def run_inference(model_, query: str, tools_str: str) -> str:
    results = run_batch_inference(model_, [{"query": query, "tools": tools_str}], batch_size=1)
    return results[0]["response_text"]

def run_inference_instrumented(model_, query: str, tools_str: str) -> dict:
    return run_batch_inference(model_, [{"query": query, "tools": tools_str}], batch_size=1)[0]

## 4. Load base model & benchmark

In [59]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.float16, device_map="auto")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

results_before = benchmark(model, test_dataset,
                           label="Base model (before fine-tuning)",
                           batch_size=8)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Memory footprint: 1.19 GB
[Base model (before fine-tuning)] Running batched inference — 1000 samples, batch_size=8


100%|██████████| 125/125 [24:04<00:00, 11.56s/it]


  Base model (before fine-tuning)
  ACCURACY
    json_valid_pct              : 100.0 %
    name_match_pct              : 100.0 %
    args_keys_match_pct         : 100.0 %
    args_exact_pct              : 100.0 %
  PERFORMANCE
    wall_total_s                : 1444.47
    avg_latency_s               : 1.3583
    p95_latency_s               : 4.3429
    avg_ttft_s                  : 0.083
    avg_tokens_per_sec          : 70.2212
    avg_tokens_generated        : 70.605
    avg_vram_delta_mb           : 0.0086
    peak_vram_reserved_mb       : 11980.0
    avg_ram_delta_mb            : 0.0006
    avg_cpu_percent             : 72.9224
    throughput_samples_per_sec  : 0.0
  CO₂  22.1697 g CO₂eq

  ✅ Results appended → benchmark_results.json


That's Impressive, model shows 100% performance on all tests (random 1000 samples from XLAM-60k-function-calling dataset from Salesforce).